# Make QSUBs for Footprinting Tools

In [1]:
from pathlib import Path
import os, re, sys, subprocess, tempfile
import numpy as np

def StartQsub(filename, nproc, memperproc, foot_program, wd):
    #add times into here
    f = open(filename, "w")
    f.write("#!/bin/bash\n\n")
    f.write("#$ -S /bin/bash\n")
    f.write(f"#$ -wd {wd}\n") 
    f.write("#$ -j y\n")                         # STDERR and STDOUT should be joined
    f.write(f"#$ -l mem_free={memperproc}G\n") 
    f.write("#$ -l scratch=25G\n")               # job requires up to 25 GiB of local /scratch space
    f.write(f"#$ -pe smp {nproc}\n")             # the job will be allotted n slots (“cores”) on a single machine
    if foot_program == "print": 
        f.write("#$ -l h_rt=100:00:00\n")        
    else: 
        f.write("#$ -l h_rt=24:00:00\n")

    if foot_program == "hint" or foot_program == "tobias":
        f.write("#$ -l x86-64-v=4\n")            # breaks on certain os idk why
    
    f.write("set -e\n")
    f.write("\n")
    f.write("module load CBI miniforge3\n")
    
    if foot_program == "hint" or foot_program == "tobias":
        f.write("conda activate foot_progs\n")
        
    if foot_program == "macs3":
        f.write("conda activate macs\n")
    
    if foot_program == "macs3" or foot_program == "print" or foot_program == "pwm":
        f.write("module load r\n")
    
    f.write("\n")
    f.write("start=$(date +%s)\n")
    return f


def CloseSub(f, fnames, server):
    if (server == "wynton"): f.write('[[ -n "$JOB_ID" ]] && qstat -j "$JOB_ID"\n')

    f.write('echo "Elapsed Time: $(($end-$start)) seconds"\n')
    tmp = '{ sum += $5 } END{ print sum }'
    f.write(f"fsize=$(ls -l {' '.join(fnames)} | awk '{tmp}' )\n")
    f.write('echo "InputFileSize: $fsize"\n')
    
    f.close()
    return


def find_bam_files(directory):
    bam_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith(".bam"):
                subdirectory = os.path.basename(root)
                pattern = re.compile(r'_(.*?)\.') #"file_interestingpart.suffix"
                group = pattern.search(file).group(1)
                cell_line = re.sub(r'^([^_]*)_.*', r'\1', file)
                bam_files.append((subdirectory, file, cell_line))
    return bam_files

In [2]:
# foot_program = "hint"
# maxvmem = 10 #GB
# nproc = 1
# memperproc = round(maxvmem/nproc)

foot_program = "tobias"
maxvmem = 16 #GB
nproc = 8
memperproc = round(maxvmem/nproc)

# foot_program = "print"
# maxvmem = 300
# nproc = 15
# memperproc = round(maxvmem/nproc)

# foot_program = "pwm"
# nproc = 1
# memperproc = 50

# foot_program = "macs3"
# nproc = 5
# memperproc = 10

In [ ]:
# #original samples
# data_dir = "/03_filtered_bams/"
# out_dir = f"/05_footprinting/01_original/{foot_program}/"
# bam_file_dict=[("","GM12878-cellFilt.bam", "GM12878"),
#                 ("","HEPG2-cellFilt.bam", "HEPG2"),
#                 ("","K562-cellFilt.bam", "K562"),
#                 ("","MCF7-cellFilt.bam", "MCF7"),
#                 ("","SKNSH-cellFilt.bam", "SKNSH")]

#cell downsampling
data_dir = "/04_downsampling/02_cells/"
out_dir = f"/05_footprinting/02_cells/{foot_program}/"
bam_file_dict = find_bam_files(data_dir)

# #read downsampling
# data_dir = "/04_downsampling/03_reads/"
# out_dir = f"/05_footprinting/03_reads/{foot_program}/"
# bam_file_dict = find_bam_files(data_dir)

# #frip downsampling
# data_dir = "/04_downsampling/04_frip/"
# out_dir = f"/05_footprinting/04_frip/{foot_program}/"
# bam_file_dict = find_bam_files(data_dir)

# #cell sim sampling
# data_dir = "/04_downsampling/06_cellsim/"
# out_dir = f"/05_footprinting/06_cellsim/{foot_program}/"
# bam_file_dict = find_bam_files(data_dir)

In [ ]:
print(bam_file_dict)        # does this look accurate?
remove_files = False        # do you want to delete the intermediate files?
sge_ncpu = "${NSLOTS:-1}"   # gen variable needed
Path(f"{out_dir}/qsubs/").mkdir(parents=True, exist_ok=True) #make qsub directory

In [6]:

for subdir, bam_file, cell_line in bam_file_dict:
    prefix = re.sub('.bam', '', bam_file, count=1)
    bamfile_path = f"{data_dir}/{subdir}/{bam_file}"
    frag_path = f"{data_dir}/{subdir}/{prefix}.frags.tsv.bgz"
    qsub_script = fr"{out_dir}/qsubs/{prefix}_{foot_program.upper()}.qsub.sh"

    f = StartQsub(qsub_script, nproc, memperproc, foot_program, wd = f"{out_dir}/qsubs/")
    
    if cell_line == "combo": peakfile_path = f"/03_peakcalls/Union_filt_500bp.exclusion.bed" #cell sim uses union peak file only.
    else: peakfile_path = f"/03_peakcalls/{cell_line}_filt_500bp.exclusion.bed"              #otherwise use cell-line specific input

    #REAL JASPAR INPUTS
    motif_dir = "/program_input_files/jaspar2022_motifs/"
    motif_file = "/program_input_files/JASPAR2022_CORE_vertebrates_non-redundant_pfms_jaspar_nameFIX.txt"

    # #SIMULATED PWM INPUTS
    # motif_dir = "/09_simPWMs/00_constructing_PWMs/final_simPWMs/"
    # motif_file = "/09_simPWMs/00_constructing_PWMs/Simulated_PFMs_jaspar.txt"

    if foot_program == "hint":

        command = f'''rgt-hint footprinting \\
        --atac-seq \\
        --organism=hg38 \\
        --paired-end \\
        --output-location={out_dir} \\
        --output-prefix={prefix} \\
        {bamfile_path} \\
        {peakfile_path}\n\n'''
        f.write(command)

        command = f'''rgt-motifanalysis matching \\
        --organism=hg38 \\
        --motif-dbs {motif_dir} \\
        --input-files {out_dir}/{prefix}.bed \\
        --output-location {out_dir}\n\n'''
        f.write(command)

        f.write("end=$(date +%s)\n")
        CloseSub(f=f, fnames=[bamfile_path, peakfile_path], server=server)

    elif foot_program == "tobias":
        genome = "/pollard/data/projects/aseveritt/refdata-cellranger-arc-GRCh38-2020-A-2.0.0/fasta/genome.fa" 
        exclusion_list = "/pollard/data/projects/aseveritt/refdata-cellranger-arc-GRCh38-2020-A-2.0.0/hg38-blacklist.v2.bed"

        #run tn5 corrrection w/ default parameters. 
        command = f'''TOBIAS ATACorrect \\
        --bam {bamfile_path} \\
        --peaks {peakfile_path} \\
        --genome {genome} \\
        --blacklist {exclusion_list} \\
        --outdir {out_dir} \\
        --cores {sge_ncpu}\n\n''' 
        f.write(command)

        #for the corrected bigwig, calculate the footprint scores per bp
        command = f'''TOBIAS FootprintScores \\
        --signal {out_dir}/{prefix}_corrected.bw \\
        --regions {peakfile_path} \\
        --output {out_dir}/{prefix}_corrected_ftscore.bw \\
        --cores {sge_ncpu}\n\n''' 
        f.write(command)

        #finally, call actual TFBS using Jaspar db. 
        command = f'''TOBIAS BINDetect \\
        --skip-excel \\
        --motifs {motif_file} \\
        --signals {out_dir}/{prefix}_corrected_ftscore.bw \\
        --genome {genome} \\
        --peaks {peakfile_path} \\
        --outdir {out_dir}/{prefix}_bindetect/ \\
        --cores {sge_ncpu}\n\n''' 
        f.write(command)

        f.write("end=$(date +%s)\n")

        #becuase the output file structure involves ~many~ subdirectories, many of which are empty... we need to collapse all the info we need. 
        #for all subdirectories that contain a *_overview.txt file, remove the header and concatenate them into one file. 
        command = f"cat <( tail -n +2 -q {out_dir}/{prefix}_bindetect/*/*_overview.txt) | awk /./ > {out_dir}/{prefix}_bindetect_TF_overviews.txt\n\n"
        f.write(command)

        f.write(f"cp {out_dir}/{prefix}_bindetect/bindetect_results.txt {out_dir}/{prefix}_bindetect_results.txt\n")
        
        if (remove_files):
            f.write(f"rm {out_dir}/{prefix}_uncorrected.bw\n")
            f.write(f"rm {out_dir}/{prefix}_bias.bw\n")
            f.write(f"rm {out_dir}/{prefix}_expected.bw\n")
            f.write(f"rm {out_dir}/{prefix}_atacorrect.pdf\n")
            f.write(f"rm {out_dir}/{prefix}_AtacBias.pickle\n")
            f.write(f"rm -r {out_dir}/{prefix}_bindetect/ \n")

        CloseSub(f=f, fnames = [bamfile_path, peakfile_path], server=server)

    elif foot_program == "print":
        if cell_line == "combo": peakRdata = f"/pollard/data/projects/aseveritt/encode_snatacseq/05_footprinting/program_input_files/peakRData/Union.printinput.RData"
        else: peakRdata = f"/pollard/data/projects/aseveritt/encode_snatacseq/05_footprinting/program_input_files/peakRData/{cell_line}.printinput.RData"
        
        scale = 30

        command = f'''Rscript /05_scripts/PRINT_FT.R \\
        --frag {frag_path} \\
        --peak {peakRdata} \\
        --motif {motif_file} \\
        --scale {scale} \\
        --outdir {out_dir}/{prefix}/ \\
        --nproc {sge_ncpu}\n\n'''
        f.write(command)
        f.write("end=$(date +%s)\n")

        f.write(f"cp {out_dir}/{prefix}/{cell_line}_granges.bed {out_dir}/{prefix}_granges.bed \n")

        if (remove_files):
            #f.write(f"rm -r {out_dir}/{prefix}/chunkedCountTensor/ \n")
            f.write(f"rm -r {out_dir}/{prefix}/chunkedFootprintResults/ \n")
            f.write(f"rm -r {out_dir}/{prefix}/chunkedTFBSResults/ \n")

        CloseSub(f=f, fnames=[frag_path], server=server)
        
    
    elif foot_program == "macs3":
        command = f'''Rscript /03_scripts/MACS_peaks.R \\
        -b {bamfile_path} \\
        -o {out_dir} \\
        --cores {sge_ncpu}\n\n'''
        f.write(command)
        f.write("end=$(date +%s)\n")
        
        if (remove_files):
            f.write(f"rm -r {out_dir}/{prefix}_peaks.xls\n")
            f.write(f"rm -r {out_dir}/{prefix}_peaks.narrowPeak\n")
            
        CloseSub(f=f, fnames=[], server=server)
        
     
    elif foot_program == "pwm":
        #if downsampled condition
        peak_dir = re.sub('pwm/', 'macs3/', out_dir, count=1)             #pwm matches are run on the macs3 calls. 
        curr_peak_file=f'{peak_dir}/{prefix}_filt_500bp.exclusion.bed'

        #if original
        #peak_dir = "/pollard/data/projects/aseveritt/encode_snatacseq/03_peakcalls/" #for original 
        #curr_peak_file=f'{peak_dir}/{cell_line}_filt_500bp.exclusion.bed'

        command = f'''Rscript /pollard/data/projects/aseveritt/encode_snatacseq/05_scripts/PWM_FT.R \\
        --peak {curr_peak_file} \\
        --motif {motif_file} \\
        --outdir {out_dir}/ \n\n'''
        f.write(command)
        f.write("end=$(date +%s)\n")

        CloseSub(f=f, fnames=[curr_peak_file], server=server)
 